In [ ]:
from cmxflow import Block, SinkBlock, SourceBlock, Workflow, parameter

In [ ]:
def reader(x):
    for i in range(10):
        yield i

def writer(x, y):
    for i in x:
        print(i)

class SimilarityBlock(Block):
    def __init__(self):
        super().__init__(input_files=["reference"])
        self.mutable(parameter.Categorical("fingerprint", "ecfp", ["ecfp", "rdk"]))

    def forward(self, x):
        return x ** 2

class SubstructureMatchBlock(Block):
    def __init__(self):
        super().__init__(input_text=["smarts"])
        self.mutable(parameter.Categorical("stereochemistry", True, [True, False]))

    def forward(self, x):
        return x ** 2

class MergeScoreBlock(Block):
    def forward(self, x):
        return x ** 2
    def check_input(self, x):
        return isinstance(x, int)

i = SourceBlock(reader)
b1 = SimilarityBlock()
b2 = SubstructureMatchBlock()
b3 = MergeScoreBlock()
o = SinkBlock(writer)
w = Workflow()
w.add(i, b1, b2, b3, o)
w

In [ ]:
w.get_required_input()

In [ ]:
required_input = {
    '1.file@reference': "test.csv",
    '2.text@smarts': "CCC",
}

w.set_required_input(required_input)
w

In [ ]:
w('a', 'b')

In [1]:
from pathlib import Path

from cmxflow import Workflow
from cmxflow.operators import MoleculeSimilarityBlock
from cmxflow.sinks import MoleculeSinkBlock
from cmxflow.sources import MoleculeSourceBlock

i = MoleculeSourceBlock()
b = MoleculeSimilarityBlock()
o = MoleculeSinkBlock()
w = Workflow()
w.add(i,b,o)
w

                 ┌────────────────┐                  
                 │ MoleculeSource │                  
                 ├────────────────┤                  
                 │ input: [FILE]  │                  
                 └────────────────┘                  
                          ↓                          
┌─────────────────────────────┐                      
│     MoleculeSimilarity      │                      
├─────────────────────────────┤   ┌─────────────────┐
│ fingerprint_type: morgan    │   │  RequiredInput  │
│ similarity_metric: tanimoto │ ← ├─────────────────┤
│ radius: 2                   │   │ queries: [FILE] │
│ nbits: 2048                 │   └─────────────────┘
└─────────────────────────────┘                      
                          ↓                          
                 ┌────────────────┐                  
                 │  MoleculeSink  │                  
                 ├────────────────┤                  
                 │ output: [

In [2]:
w.set_required_input({'1.file@queries': 'test_query.csv'})
w

                     ┌────────────────┐                      
                     │ MoleculeSource │                      
                     ├────────────────┤                      
                     │ input: [FILE]  │                      
                     └────────────────┘                      
                              ↓                              
┌─────────────────────────────┐                              
│     MoleculeSimilarity      │                              
├─────────────────────────────┤   ┌─────────────────────────┐
│ fingerprint_type: morgan    │   │      RequiredInput      │
│ similarity_metric: tanimoto │ ← ├─────────────────────────┤
│ radius: 2                   │   │ queries: test_query.csv │
│ nbits: 2048                 │   └─────────────────────────┘
└─────────────────────────────┘                              
                              ↓                              
                     ┌────────────────┐                      
        

In [3]:
w.blocks[1].get_params()['fingerprint_type'].set('rdkit')
w.blocks[1].get_params()['similarity_metric'].set('cosine')
w

                    ┌────────────────┐                     
                    │ MoleculeSource │                     
                    ├────────────────┤                     
                    │ input: [FILE]  │                     
                    └────────────────┘                     
                             ↓                             
┌───────────────────────────┐                              
│    MoleculeSimilarity     │                              
├───────────────────────────┤   ┌─────────────────────────┐
│ fingerprint_type: rdkit   │   │      RequiredInput      │
│ similarity_metric: cosine │ ← ├─────────────────────────┤
│ radius: 2                 │   │ queries: test_query.csv │
│ nbits: 2048               │   └─────────────────────────┘
└───────────────────────────┘                              
                             ↓                             
                    ┌────────────────┐                     
                    │  MoleculeSink  │  

In [4]:
w(Path('test.csv'), Path('test-out.csv'))